# TRINETRA · Account A · phase2 (FAST) — fire CNN + colour baseline + video

**Kaise chalayein (teammate ke liye, 1 minute):**
1. Kaggle → *Create → Import Notebook* → ye file upload karo.
2. Right panel → **Add Input** → `happyparsi/trinetra-dataset` (mounted hota hai, download nahi).
3. **Settings → Accelerator → GPU T4 x2**, **Internet ON**.
4. **Save Version → Save & Run All (commit)**. Tab band kar sakte ho, run chalta rahega.

**Purane A_phase2 se kya alag hai:**
- GPU na mile to notebook *turant* ruk jaata hai (CPU par ghanton nahi chalega).
- Dataset (61 GB, unpacked folders) ko poora scan **nahi** karta: ek tez finder sirf A ke 8 folders dhoondhta hai (sirf folder listing, images nahi gin-ta), phir sirf fire sets convert hote hain.
- Har command ka log *live* aata hai, har 5 min chup rehne par `[heartbeat]`, aur har stage ka `[time]`.
- Har stage par timeout: atka to wo stage band, baaki results phir bhi save.

Expected total (T4 x2): **~25–50 min** (planning estimate). Output: `out_A_phase2.zip` + `out/results/RESULTS.md`.
Numbers = OUR RESULTS on a prototype subset; full-benchmark papers se direct compare mat karna.

In [ ]:
ACCOUNT = "A"
PHASE   = "phase2"   # fast check: 1 seed, 5 epochs
DATASET = "/kaggle/input/trinetra-dataset"
READY   = "/kaggle/input/trinetra-ready-v1/npz"   # agar teammate ne converted NPZ publish kiye hain to conversion skip
GITHUB  = ""
import os
os.environ.update(
    BUILD_PARTS="fire",                 # A ko sirf fire + held-out + confusers + FireSense chahiye
    STAGE_SKIP_REGEX="cifake|deepfake|stylegan|face-images|ai-generated|aiart|landslide4sense|sen1floods|hephaestus|insar|crisismmd|avenue|eurosat|glof|imerg|worldcover|landcover",
    PYTHONUNBUFFERED="1")              # converter output turant log me dikhe
TIMEOUT_MIN = {"find": 7, "build": 60, "fire": 120, "video": 45}   # stage atke to itne minute baad band

In [ ]:
import base64, io, zipfile, os
_Z = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(_Z))).extractall('/kaggle/working')
print('embedded TRINETRA code unpacked:', os.path.isdir('/kaggle/working/TRINETRA/trinetra/kaggle'))

In [ ]:
# 0) GPU check: fail fast instead of training a CNN on CPU for hours
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU NAHI MILA → right panel: Settings → Accelerator → GPU T4 x2 chuno, phir Save & Run All. "
                     "(CPU par fire CNN ghanton chalega — isi wajah se purana run 4 ghante atka ho sakta hai)")
print("GPU OK:", torch.cuda.get_device_name(0), "x", torch.cuda.device_count())

In [ ]:
import subprocess, shutil, glob, pathlib, time, threading, signal
def sh(cmd, timeout_min=None, tag=None):
    """Run a shell command with live output, a heartbeat every ~5 silent minutes and an optional hard timeout."""
    tag = tag or cmd.split()[0]; print("$", cmd, flush=True); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    last = [time.time()]
    def beat():
        while p.poll() is None:
            time.sleep(30)
            if p.poll() is None and time.time() - last[0] > 290:
                print(f"[heartbeat] {tag} still running · {(time.time() - t0) / 60:.0f} min", flush=True); last[0] = time.time()
    threading.Thread(target=beat, daemon=True).start()
    def kill():
        print(f"[timeout] {tag} ran > {timeout_min} min → stopping it (partial outputs are kept)", flush=True)
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
    timer = threading.Timer(timeout_min * 60, kill) if timeout_min else None
    if timer: timer.start()
    for line in p.stdout:
        print(line, end="", flush=True); last[0] = time.time()
    rc = p.wait()
    if timer: timer.cancel()
    print(f"[time] {tag}: {(time.time() - t0) / 60:.1f} min · exit {rc}", flush=True)
    return rc
# 1) locate code
hits = glob.glob("/kaggle/input/**/trinetra/kaggle/run_hackathon.sh", recursive=True)
CODE = "/kaggle/working/TRINETRA"
if os.path.isdir(CODE + "/trinetra/kaggle"):
    pass  # embedded package already unpacked
elif hits:
    shutil.copytree(str(pathlib.Path(hits[0]).parents[2]), CODE, dirs_exist_ok=True)
elif GITHUB:
    sh(f"git clone --depth 1 {GITHUB} /kaggle/working/repo && cp -r /kaggle/working/repo/TRINETRA {CODE}")
else:
    raise SystemExit("Attach the TRINETRA code dataset (or set GITHUB)")
print("code at", CODE); T_RUN = time.time()

In [ ]:
EPOCHS, SEEDS = ("5", "0") if PHASE == "phase2" else ("15", "0 1 2")
os.environ.update(OUT="/kaggle/working/out", EPOCHS=EPOCHS, SEEDS=SEEDS, PY="python",
                  KAGGLE_INPUT=DATASET, DFIRE_N="3000" if PHASE == "phase2" else "4000")
sh("pip -q install timm", tag="pip")
sh("nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv || true", tag="nvidia-smi")

In [ ]:
# ---- fast A-only dataset finder (lists folders only; never walks image files)
import os, re, time
A_KEYS = ["DFIRE", "DEEPFIRE", "FIREDS", "WEATHER", "CANDLE", "SCENES", "FIRESENSE", "HOMEFIRE"]
LEAF_NAMES = {"images", "labels", "img", "imgs", "frames", "jpegimages"}
def find_a_roots(base, max_depth=6, budget_s=420, skip_regex=""):
    skip = re.compile(skip_regex, re.I) if skip_regex else None
    t0, found, q, n = time.time(), {}, [(base, 0)], 0
    def add(k, p):
        if k not in found:
            found[k] = p; print(f"  found {k:9s} = {p}   ({time.time() - t0:.0f}s)", flush=True)
    while q and time.time() - t0 < budget_s and len(found) < len(A_KEYS):
        d, dep = q.pop(0)
        try:
            with os.scandir(d) as it: ents = list(it)
        except OSError:
            continue
        n += 1
        if n % 200 == 0: print(f"  … {n} folders scanned ({time.time() - t0:.0f}s)", flush=True)
        dirs = sorted(e.name for e in ents if e.is_dir(follow_symlinks=False))
        files = {e.name.lower() for e in ents if not e.is_dir(follow_symlinks=False)}
        low, pl, name = {x.lower() for x in dirs}, d.lower(), os.path.basename(d).lower()
        if "data.yaml" in files and "home" not in pl and "flame" not in pl: add("DFIRE", d)
        if name == "training" and {"fire", "nofire"} <= low: add("DEEPFIRE", os.path.dirname(d))
        if {"fire_images", "non_fire_images"} <= low: add("FIREDS", d)
        if {"fogsmog", "sandstorm", "lightning", "rain"} <= low: add("WEATHER", d)
        if name == "train" and "images" in low and "flame" in pl: add("CANDLE", os.path.dirname(os.path.dirname(d)))
        if {"street", "buildings", "forest", "mountain"} <= low and not name.startswith(("test", "pred")): add("SCENES", d)
        if {"fire", "smoke"} <= low and all(os.path.isdir(os.path.join(d, h, k)) for h in ("fire", "smoke") for k in ("pos", "neg")): add("FIRESENSE", d)
        if {"images", "labels"} <= low and "home" in pl: add("HOMEFIRE", os.path.dirname(d) if name in ("train", "test") else d)
        if dep < max_depth and len(files) < 300:          # image-heavy folders are leaves: never descend into them
            for x in dirs:
                p = os.path.join(d, x)
                if x.lower() in LEAF_NAMES or (skip and skip.search(p)): continue
                q.append((p, dep + 1))
    print(f"[find] {n} folders scanned in {time.time() - t0:.0f}s · found {len(found)}/{len(A_KEYS)}", flush=True)
    return found
def show_tree(base, depth=2):
    for root, dirs, files in os.walk(base):
        lvl = root[len(base):].count(os.sep)
        print("  " * lvl + os.path.basename(root) + f"/  ({len(files)} files)")
        if lvl >= depth - 1: dirs[:] = []
        else: dirs[:] = sorted(dirs)[:40]

# 2) data: mounted team dataset (unpacked folders) → find only A's roots → convert only the fire sets
if not os.path.isdir(DATASET):
    cands = [d.rstrip("/") for d in glob.glob("/kaggle/input/*/") + glob.glob("/kaggle/input/*/*/*/") if "trinetra-dataset" in d.lower()]
    if cands: DATASET = cands[0]
    else:
        print("Attached inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")
        raise SystemExit("DATASET NOT ATTACHED: right panel → Add Input → happyparsi/trinetra-dataset → Add, phir Run All.")
print("dataset at", DATASET, flush=True); os.environ["KAGGLE_INPUT"] = DATASET
if os.path.isdir(READY) and glob.glob(READY + "/*.npz"):
    os.environ["DATA"] = READY; print("using ready NPZ pack:", READY)
else:
    os.environ["DATA"] = "/kaggle/working/npz"; os.makedirs(os.environ["DATA"], exist_ok=True)
    roots = find_a_roots(DATASET, budget_s=TIMEOUT_MIN["find"] * 60, skip_regex=os.environ.get("STAGE_SKIP_REGEX", ""))
    if "DFIRE" not in roots:
        print("\nDataset ke top folders (ye AI ko bhejo):"); show_tree(DATASET, depth=3)
        raise SystemExit("D-Fire (data.yaml wala folder) nahi mila. Upar ka tree AI ko bhejo.")
    os.environ.update(roots)
    missing = [k for k in A_KEYS if k not in roots]
    if missing: print("[warn] nahi mile (ye parts skip honge, fire training phir bhi chalegi):", ", ".join(missing))
    sh(f"set -o pipefail; bash {CODE}/trinetra/kaggle/run_hackathon.sh build 2>&1 | tee /kaggle/working/build.log",
       timeout_min=TIMEOUT_MIN["build"], tag="build")
sh("ls -la $DATA; df -h /kaggle/working | tail -1", tag="ls")
if not os.path.exists(os.environ["DATA"] + "/fire.npz"):
    sh("tail -40 /kaggle/working/build.log", tag="build-tail")
    raise SystemExit("fire.npz nahi bana. Upar build.log ki aakhri 40 lines AI ko bhejo.")

In [ ]:
# 3) train / evaluate; results collected after every stage so a timeout keeps partial outputs
R = f"bash {CODE}/trinetra/kaggle/run_hackathon.sh"
status = {}
for stage in ["fire", "video"]:
    status[stage] = sh(f"set -o pipefail; {R} {stage} 2>&1 | tee /kaggle/working/{stage}.log", timeout_min=TIMEOUT_MIN[stage], tag=stage)
    sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results > /dev/null 2>&1", tag="collect")
print("stage exit codes:", status, f"· total {(time.time() - T_RUN) / 60:.0f} min")
open("/kaggle/working/stage_status.json", "w").write(__import__("json").dumps(status, indent=2))

In [ ]:
# 4) collect + package
sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results", tag="collect")
sh("cp /kaggle/working/*.log /kaggle/working/stage_status.json /kaggle/working/out/ 2>/dev/null; cp -r /kaggle/working/trinetra_stage/*.json /kaggle/working/out/ 2>/dev/null", tag="copy")
shutil.make_archive(f"/kaggle/working/out_{ACCOUNT}_{PHASE}", "zip", "/kaggle/working/out")
sh("cat /kaggle/working/out/results/RESULTS.md 2>/dev/null | head -80", tag="results")